In [130]:
import pandas as pd, numpy as np

df = pd.read_csv('data/combined.csv')
# print(df.shape)
# df.columns

In [131]:
df['Date_x'] = pd.to_datetime(df['Date_x']).dt.date
df['daily_change_y'] = pd.to_numeric(df['daily_change_y'], errors='coerce')
df = df.dropna(subset=['daily_change_y']).copy()
df['daily_change'] = df['daily_change_x']
df['EPS_est'] = df['EPS Estimate']
df['w1_change'] = df['1w_change']
df['m1_change'] = df['1m_change']
df['m3_change'] = df['3m_change']
df['y1_change'] = df['1yr_change']
df['wk52_high'] = df['52wk_high']
df['wk52_low'] = df['52wk_low']

if 'Surprise(%)' in df.columns:
    df["Surprise(%)"] = pd.to_numeric(df["Surprise(%)"], errors="coerce")
    df = df.sort_values(["ticker","Date_x"])
    df["surprise_shift1"] = df.groupby("ticker")["Surprise(%)"].shift(1)

# Drop leaky columns
for col in ["Reported EPS","Surprise(%)","Adj Close", 'daily_change_x', 'EPS Estimate', "1w_change", 
            "1m_change", "3m_change", "1yr_change", "52wk_high", "52wk_low"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

df["high_low_range"] = (df["High"] - df["Low"]) / df["Open"]
# if "sma20" in df.columns:
#     df["open_sma20_ratio"] = df["Open"] / df["sma20"]
df["vol_log"] = np.log1p(df["Volume"].fillna(0))

df['y'] = (df['daily_change_y'] >= 0.03).astype(int)
df.shape

(20501, 42)

In [37]:
df.columns

Index(['Earnings Date', 'ticker', 'Date_x', 'Quarter', 'Close', 'High', 'Low',
       'Open', 'Volume', 'sma20', 'sma50', 'sma200', 'sma20_change',
       'sma50_change', 'sma200_change', 'rsi_14', 'volatility_wk',
       'volatility_1m', 'next_trading_date', 'Date_y', 'daily_change_y',
       'shifted_rep_eps', 'TTM_EPS', 'PE_Ratio', 'Forward_PE', 'Ticker',
       'Sector', 'Industry', 'HeadquartersCity', 'HeadquartersState',
       'daily_change', 'EPS_est', 'w1_change', 'm1_change', 'm3_change',
       'y1_change', 'wk52_high', 'wk52_low', 'surprise_shift1',
       'high_low_range', 'open_sma20_ratio', 'vol_log', 'y'],
      dtype='object')

In [132]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, precision_score, recall_score


num_feats = ['EPS_est', 'TTM_EPS', 'PE_Ratio', 'Forward_PE', 'Open',
             'High', 'Low', 'Volume', 'daily_change', 'w1_change', 'm1_change', 'm3_change',
             'y1_change', 'wk52_high', 'wk52_low', 'sma20_change', 'sma50_change',
             'sma200_change', 'rsi_14', 'volatility_wk', 'volatility_1m', 
             'shifted_rep_eps', 'surprise_shift1', 'high_low_range', 'vol_log']
num_feats = [f for f in num_feats if f in df.columns]

cat_feats = ['Quarter', 'Sector', 'Industry', 'HeadquartersState']
cat_feats = [f for f in cat_feats if f in df.columns]

df = df.sort_values("Date_x")
df = df[num_feats + cat_feats + ['y']].dropna().copy()
# y = df.pop('y')
y = df['y']
# df.dropna().shape ## 16812, 28

x_train, x_test, y_train, y_test = train_test_split(df[num_feats + cat_feats], y, test_size=0.2, shuffle=False)
numeric_transform = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_transform = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer([("num", numeric_transform, num_feats),
                                  ("cat", cat_transform, cat_feats)])


In [ ]:
# import xgboost as xgb
# xgb_params = {
#     'n_estimators': 5000,
#     'learning_rate': 0.01,
#     'booster': 'gbtree',
#     'max_depth': 6,
#     'objective': 'binary:logistic',
#     'eval_metric': 'aucpr',
#     'early_stopping_rounds': 250
# }

# models = {
#     'LosgisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced')
#     # 'RandomForest': RandomForestClassifier(n_estimators=1000, n_jobs=-1, class_weight='balanced'),
#     # 'GradientBoosting': GradientBoostingClassifier(n_estimators=1000),
#     # 'xgboost': xgb.XGBClassifier(**xgb_params)
# }

# trained = {}
# results = {}
# for name, model in models.items():
#     print(f'Now training: {name}')
#     pipe = Pipeline([('pre', preprocessor), ('clf', model)])
#     if name == 'xgboost':
#         x_test_pre = preprocessor.transform(x_test)
#         # Only XGBoost supports eval_set
#         pipe.fit(x_train, y_train, clf__eval_set=[(x_test_pre, y_test)], clf__verbose=100)
#     else:
#         pipe.fit(x_train, y_train)

#     prob = pipe.predict_proba(x_test)[:, 1]
#     pred = (prob >= 0.5).astype(int)
#     roc = roc_auc_score(y_test, prob)
#     precision, recall, _ = precision_recall_curve(y_test, prob)
#     pr_auc = auc(recall, precision)
#     f105 = f1_score(y_test, pred)
#     # Optimize threshold for F1
#     thresholds = np.linspace(0.1, 0.9, 50)
#     best_f1, best_thr, best_prec, best_rec = 0, 0.5, 0, 0
#     for thr in thresholds:
#         pred_thr = (prob >= thr).astype(int)
#         f1 = f1_score(y_test, pred_thr)
#         if f1 > best_f1:
#             best_f1 = f1
#             best_thr = thr
#             best_prec = precision_score(y_test, pred_thr)
#             best_rec = recall_score(y_test, pred_thr)
#     results[name] = {
#         'roc_auc': roc, 
#         'pr_auc': pr_auc, 
#         'f1_0.5': f105,
#         'best_f1': best_f1,
#         'best_threshold': best_thr,
#         'prec@best': best_prec,
#         'rec@best': best_rec}
#     trained[name] = pipe

# results

Now training: LosgisticRegression


{'LosgisticRegression': {'roc_auc': 0.6258875642931477,
  'pr_auc': 0.2697236020038265,
  'f1_0.5': 0.37505904581955596,
  'best_f1': 0.3926838372527062,
  'best_threshold': np.float64(0.44285714285714284),
  'prec@best': 0.2659251769464105,
  'rec@best': 0.7503566333808844}}

In [24]:
state_counts = df["HeadquartersState"].value_counts(dropna=True)
big_states = state_counts[state_counts >= 500].index.tolist()
# print(big_states)
state_results = {}
for state in big_states:
    df_st = df[df['HeadquartersState'] == state]
    y = df_st.pop('y')
    # df_sd = df.drop(columns=['y']).copy()
    x_train, x_test, y_train, y_test = train_test_split(df_st[num_feats + cat_feats], y, test_size=0.2, shuffle=False)
    
    if len(x_train) < 50 or len(x_test) < 30:
        continue
    pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    pipe.fit(x_train, y_train)
    prob = pipe.predict_proba(x_test)[:,1]
    pred = (prob >= 0.5).astype(int)
    roc = roc_auc_score(y_test, prob)
    f105 = f1_score(y_test, pred)
    state_results[state] = {"train_n": len(x_train), "test_n": len(x_test), "roc_auc": roc, 'f1_0.5': f105}
    
state_results

{'California': {'train_n': 1714,
  'test_n': 429,
  'roc_auc': 0.6311726643205812,
  'f1_0.5': 0.2598870056497175},
 'Texas': {'train_n': 1369,
  'test_n': 343,
  'roc_auc': 0.6541705282669138,
  'f1_0.5': 0.3981042654028436},
 'New York': {'train_n': 1063,
  'test_n': 266,
  'roc_auc': 0.6309157728943223,
  'f1_0.5': 0.44171779141104295},
 'Illinois': {'train_n': 857,
  'test_n': 215,
  'roc_auc': 0.653817365269461,
  'f1_0.5': 0.40625},
 'Ohio': {'train_n': 679,
  'test_n': 170,
  'roc_auc': 0.5769927536231885,
  'f1_0.5': 0.34615384615384615},
 'Pennsylvania': {'train_n': 594,
  'test_n': 149,
  'roc_auc': 0.6310328185328186,
  'f1_0.5': 0.4044943820224719},
 'Virginia': {'train_n': 542,
  'test_n': 136,
  'roc_auc': 0.6062893081761006,
  'f1_0.5': 0.38095238095238093},
 'Georgia': {'train_n': 536,
  'test_n': 134,
  'roc_auc': 0.5990147783251232,
  'f1_0.5': 0.3783783783783784},
 'Florida': {'train_n': 493,
  'test_n': 124,
  'roc_auc': 0.5607725607725608,
  'f1_0.5': 0.325},
 'Mas

In [ ]:
# States to build individual models for
# California, Texas, New York, Illinois,
# Pennsylvania, Michigan

['California', 'Texas', 'New York', 'Illinois', 'Ohio', 'Pennsylvania']

In [25]:
sc_counts = df["Sector"].value_counts(dropna=True)
big_sector = sc_counts[sc_counts >= 500].index.tolist()
# print(big_states)
sector_results = {}
for sector in big_sector:
    df_sc = df[df['Sector'] == sector]
    y = df_sc.pop('y')
    # df_sd = df.drop(columns=['y']).copy()
    x_train, x_test, y_train, y_test = train_test_split(df_sc[num_feats + cat_feats], y, test_size=0.2, shuffle=False)
    
    if len(x_train) < 50 or len(x_test) < 30:
        continue
    pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    pipe.fit(x_train, y_train)
    prob = pipe.predict_proba(x_test)[:,1]
    roc = roc_auc_score(y_test, prob)
    pred = (prob >= 0.5).astype(int)
    f105 = f1_score(y_test, pred)
    sector_results[sector] = {"samples": len(x_train) + len(x_test), "roc_auc": roc, 'f1_0.5': f105}
    
sector_results

{'Financials': {'samples': 2310,
  'roc_auc': 0.6911391559829061,
  'f1_0.5': 0.37333333333333335},
 'Technology': {'samples': 2024,
  'roc_auc': 0.63791460160122,
  'f1_0.5': 0.3404255319148936},
 'Energy': {'samples': 1693,
  'roc_auc': 0.5651969981238274,
  'f1_0.5': 0.22950819672131148},
 'Health Care': {'samples': 1392,
  'roc_auc': 0.6705998681608438,
  'f1_0.5': 0.41450777202072536},
 'Retailing': {'samples': 1316,
  'roc_auc': 0.6274353241775791,
  'f1_0.5': 0.38461538461538464},
 'Business Services': {'samples': 994,
  'roc_auc': 0.5986842105263158,
  'f1_0.5': 0.4065040650406504},
 'Industrials': {'samples': 880,
  'roc_auc': 0.5565217391304348,
  'f1_0.5': 0.384},
 'Materials': {'samples': 779,
  'roc_auc': 0.614167650531287,
  'f1_0.5': 0.34285714285714286},
 'Transportation': {'samples': 708,
  'roc_auc': 0.6137218045112782,
  'f1_0.5': 0.3170731707317073},
 'Food, Beverages & Tobacco': {'samples': 584,
  'roc_auc': 0.5869565217391305,
  'f1_0.5': 0.345679012345679},
 'Hot

In [ ]:
# Sectors to build individual models for
# Financials, Technology, Health Care, Business Services

In [49]:
ind_counts = df["Industry"].value_counts(dropna=True)
big_industry = ind_counts[ind_counts >= 500].index.tolist()
# print(big_states)
industry_results = {}
for industry in big_industry:
    df_ind = df[df['Industry'] == industry]
    y = df_ind.pop('y')
    # df_sd = df.drop(columns=['y']).copy()
    x_train, x_test, y_train, y_test = train_test_split(df_ind[num_feats + cat_feats], y, test_size=0.2, random_state=31)
    
    if len(x_train) < 50 or len(x_test) < 30:
        continue
    pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    pipe.fit(x_train, y_train)
    prob = pipe.predict_proba(x_test)[:,1]
    roc = roc_auc_score(y_test, prob)
    pred = (prob >= 0.5).astype(int)
    f105 = f1_score(y_test, pred)
    industry_results[industry] = {"samples": len(x_train) + len(x_test), "roc_auc": roc, 'f1_0.5': f105}
    
industry_results

{'Utilities: Gas and Electric': {'samples': 670,
  'roc_auc': 0.5039370078740157,
  'f1_0.5': 0.10810810810810811},
 'Commercial Banks': {'samples': 650,
  'roc_auc': 0.6272189349112426,
  'f1_0.5': 0.35443037974683544},
 'Specialty Retailers: Other': {'samples': 516,
  'roc_auc': 0.6419120785317969,
  'f1_0.5': 0.5526315789473685},
 'Industrial Machinery': {'samples': 501,
  'roc_auc': 0.6143589743589744,
  'f1_0.5': 0.38095238095238093}}

In [ ]:
# Industry to build individual models for
# Specialty Retailers

In [137]:
import joblib

states = ["California", "Texas", "New York", "Illinois", "Pennsylvania", "Michigan"]
sectors = ["Financials", "Technology", "Health Care", "Business Services"]
industries = ["Specialty Retailers: Other"]

numeric_transform = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_transform = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer([("num", numeric_transform, num_feats),
                                  ("cat", cat_transform, cat_feats)])

final_df = df[num_feats + cat_feats]
general_model = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=1000))])
general_model.fit(final_df, y)
joblib.dump(general_model, 'models/clf_general.joblib')

def build_models(df, filter_col, filter_val):
    df_f = df[df[filter_col] == filter_val]
    y = df_f.pop('y')
    df_f = df_f[num_feats + cat_feats]
    model = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
    model.fit(df_f, y)   
    if filter_val == 'Specialty Retailers: Other':
        joblib.dump(model, f'models/clf_{filter_col}_Specialty_Retailers_Other.joblib')     
    else:
        joblib.dump(model, f'models/clf_{filter_col}_{filter_val}.joblib') 
    

In [134]:
for s in states:
    if "HeadquartersState" in df.columns:
        build_models(df, "HeadquartersState", s)
for s in sectors:
    if "Sector" in df.columns:
        build_models(df, "Sector", s)
for s in industries:
    if "Industry" in df.columns:
        build_models(df, "Industry", s)

In [55]:
general_model

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# pd.set_option('display.max_columns', None)

# final_df.sample(1)

,EPS_est,TTM_EPS,PE_Ratio,Forward_PE,Open,High,Low,Volume,daily_change,w1_change,m1_change,m3_change,y1_change,wk52_high,wk52_low,sma20_change,sma50_change,sma200_change,rsi_14,volatility_wk,volatility_1m,shifted_rep_eps,surprise_shift1,high_low_range,vol_log,Quarter,Sector,Industry,HeadquartersState
14037,0.82,7.27,5.647606,50.070846,39.791106,41.229567,39.781578,2372900.0,0.01245,0.039556,0.202232,0.212427,-0.069697,55.223385,30.310661,0.089154,0.161822,-0.000816,77.980783,0.40242,2.071038,1.44,-26.98,0.03639,14.679624,3,Energy,"Mining, Crude-Oil Production",Colorado


In [ ]:
# final_df = df[num_feats + cat_feats]
# final_df = preprocessor.fit_transform(df)

# final_cali = df[df['HeadquartersState'] == 'California']
# cali_y = final_cali.pop('y')
# # print(len(final_cali), len(cali_y))
# final_cali = final_cali[num_feats + cat_feats]
# final_cali = preprocessor.transform(final_cali)

# clf_gen = LogisticRegression(max_iter=1000, class_weight='balanced')
# clf_cali = LogisticRegression(max_iter=1000, class_weight='balanced')

# clf_gen.fit(final_df, y)
# clf_cali.fit(final_cali, cali_y)

2143 2143


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [ ]:
# import joblib
# joblib.dump(preprocessor, 'models/preprocessor.joblib')
# joblib.dump(clf_gen, 'models/clf_general.joblib')
# joblib.dump(clf_cali, 'models/clf_california.joblib')
# with open("data/feature_list.txt","w") as f:
#     f.write("\n".join(num_feats + cat_feats))
